## DINOv2 LSTM ##

In [ ]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm

from PIL import Image
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix


import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import Dataset, DataLoader


import os
import pickle
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from PIL import Image

## Device

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# device = torch.device("cpu")
device

## Heatmap Test

In [ ]:
import moviepy as mpy
import copy as cp
from pyskl_lib import *
import torch

my_anno = torch.load("/home/osero/Desktop/CMPE/pyskl/demo/my_anno.pth")
my_keypoint_heatmap = get_pseudo_heatmap(cp.deepcopy(my_anno))
my_keypoint_mapvis = vis_heatmaps(my_keypoint_heatmap)
my_keypoint_mapvis = [add_label(f, my_anno['frame_dir'].split('/')[-2] + '/' + my_anno['frame_dir'].split('/')[-1]) for f in my_keypoint_mapvis]
my_vid = mpy.ImageSequenceClip(my_keypoint_mapvis, fps=24)
my_vid.display_in_notebook()

## Prepare Dataset

In [ ]:
# pose_pickle_folder = '/media/osero/SamsungSSD/CMPE_SSD/mmpose-full/0001/User_2_001.pickle'
pose_pickle_folder = '/media/osero/SamsungSSD/CMPE_SSD/mmpose-full/'

def get_active_frames_from_pickle(input_raw) -> np.ndarray:
    threshold = (
        (((input_raw["pose"]["left_hip"][:, 1] + input_raw["pose"]["right_hip"][:, 1]) / 2 )* 7)
        + input_raw["pose"]["nose"][:, 1] * 3
    ) / 10

    active_frames = (
        np.minimum(
            input_raw["hand_left"]["left_lunate_bone"][:, 1],
            input_raw["hand_right"]["right_lunate_bone"][:, 1],
        )
        < threshold
    )

    active_frame_indices = np.argwhere(active_frames).squeeze()
    return active_frame_indices


def get_active_frames(label_name, sample_name):
    pickle_file_name = f"{pose_pickle_folder}/{label_name}/{sample_name}.pickle"
    file = open(pickle_file_name, 'rb')
    input_raw = pickle.load(file)

    return get_active_frames_from_pickle(input_raw)

In [ ]:
import moviepy as mpy
import copy as cp
from pyskl_lib import *
import torch

frame_frequency = 2

def create_label_dict(classes):
    label_dict = {}
    for i in range(0,len(classes)):
        label_dict[classes[i]] = i
    return label_dict

def combine_heatmaps(heatmaps):
    heatmaps = [np.max(x, axis=0) for x in heatmaps]
    return heatmaps

class CustomHeatmapDataset(Dataset):
    def __init__(self, pickle_path = "/media/osero/SamsungSSD/pickles/bsign22_heatmap_format_train.pkl"):
        pickle_file = open(pickle_path, 'rb')
        all_annotations = pickle.load(pickle_file)
        annotation_labels = [x['label'] for x in all_annotations if x['label']<50]
        annotations = all_annotations[0: len(annotation_labels)]
        annotation_paths = [x['frame_dir'] for x in all_annotations[0: len(annotation_labels)]]

        self.annotations = annotations
        self.paths = annotation_paths
        self.classes = np.unique(annotation_labels)
        label_dict = create_label_dict(self.classes)
        self.labels = [label_dict[x] for x in annotation_labels]

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        splited_paths = self.paths[idx].split('/')

        active_frame_indices = get_active_frames(splited_paths[-2],splited_paths[-1])
        active_frame_indices = (
            active_frame_indices
            if active_frame_indices.size > 10
            else np.arange(0, len(self.annotations[idx]))
        )

        keypoint_heatmaps1 = get_pseudo_heatmap(cp.deepcopy(self.annotations[idx]))
        keypoint_heatmaps2 = combine_heatmaps(keypoint_heatmaps1)
        keypoint_heatmaps = keypoint_heatmaps2[0::frame_frequency]
        np_stacked_array = np.stack(keypoint_heatmaps)
        tensor = torch.from_numpy(np_stacked_array)
        return tensor, self.labels[idx] 


In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence

# Step 2: Collate function
def collate_fn(batch):
    sequences, lengths, labels = zip(*batch)
    lengths = torch.tensor(lengths)
    labels = torch.tensor(labels)

    # Pad sequences to the maximum length in the batch
    padded_sequences = pad_sequence([torch.tensor(seq) for seq in sequences], batch_first=True)

    # Sort by lengths in descending order
    sorted_lengths, sorted_indices = lengths.sort(descending=True)
    sorted_sequences = padded_sequences[sorted_indices]
    sorted_labels = labels[sorted_indices]
    return sorted_sequences, sorted_lengths, sorted_labels


In [ ]:
train_dataset = CustomHeatmapDataset('/media/osero/SamsungSSD/pickles/bsign22_heatmap_format_train.pkl')
test_dataset = CustomHeatmapDataset('/media/osero/SamsungSSD/pickles/bsign22_heatmap_format_test.pkl')

batch_size = 1
num_workers = 4

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)  # Adjust batch size as needed
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)
class_names = train_dataset.classes
class_names

input_dim = train_dataset[0][0][0].shape # Get input dimension from a single feature from a video
num_classes = len(set(train_dataset.classes))
print("input_dim: ", input_dim, " num_classes: ", num_classes)
print("train_dataset size: ", len(train_dataset))
print("test_dataset size: ", len(test_dataset))


## Model

In [ ]:
import torchvision.models as models


class VideoClassifierLSTM(nn.Module):
    def __init__(self, hidden_dim, num_layers, num_classes):
        super(VideoClassifierLSTM, self).__init__()
        self.cnn = models.resnet18(pretrained=True)
        self.cnn.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.lstm = nn.LSTM(self.cnn.fc.in_features, hidden_dim, num_layers, batch_first=True, bidirectional=False)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)
        self.cnn.fc = nn.Identity()  # Remove final FC layer of ResNet

    def forward(self, x):
        # LSTM expects input shape: (batch, seq, features)
        # batch_size, num_frames = x.size(0), x.size(1)
        # cnn_features = []
        # for i in range(num_frames):
        #     frame = x[:, i, :, :]  # Select each frame in the batch
        #     features = self.cnn(frame)
        #     cnn_features.append(features)
        
        # # cnn_features: list of (batch_size, feature_size)
        # cnn_features = torch.stack(cnn_features, dim=1)  # (batch_size, num_frames, feature_size)

        batch_size, time_steps, height, width = x.size()
        x = x.view(batch_size * time_steps, 1, height, width)
        
        # Feature extraction
        features = self.cnn(x)
        features = features.view(batch_size, time_steps, -1)  # Reshape for LSTM
        _, (hidden, _) = self.lstm(features)  # Use last hidden state
        output = self.dropout(hidden[-1])
        output = self.fc(output)  # Take hidden state of the last LSTM layer
        return output
    
hidden_dim = 216
num_layers = 2
model = VideoClassifierLSTM(hidden_dim=hidden_dim, num_layers=num_layers, num_classes=num_classes)
model = model.to(device)

## Functions

In [ ]:
def test_images():
    correct = 0
    top_5_correct = 0
    total = 0
    running_loss = 0.0
    # since we're not training, we don't need to calculate the gradients for our outputs
    test_predicted = []
    test_labels = []

    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            labels = labels.to(device)

            # calculate outputs by running images through the network
            outputs = model(features)
            loss = criterion(outputs, labels)

            # the class with the highest energy is what we choose as prediction
            _, predicted = torch.topk(outputs.data, 1)
            _, predicted_top_5 = torch.topk(outputs.data, 5)
            total += labels.size(0)
            correct += (predicted == labels).sum().item() 
            top_5_correct += (predicted_top_5 == labels).any().sum().item()
            running_loss += loss.item()

            test_labels += (labels.cpu().numpy().tolist())
            test_predicted += (predicted.cpu().numpy().tolist())

    avg_loss = running_loss / total
    accuracy = 100 * correct / total
    top_5_accuracy = 100 * top_5_correct / total
    print(f'Accuracy of the network on the {len(test_loader)*batch_size} test video: {accuracy:.4f} %, top5: {top_5_accuracy:.4f} %, avg_loss: {avg_loss}')
    return accuracy, top_5_accuracy, avg_loss
                # print('total: ', total, ', loss: ', loss)


In [ ]:
import datetime
from time import gmtime, strftime
def get_current_time():
    return strftime("%Y-%m-%d_%H-%M-%S", gmtime())

def save_model_result(current_time):
    result_name = 'lstm_results/LSTM_Heatmap_' + current_time + '.pth'
    torch.save({'name': result_name,
                'model_state_dict': model.state_dict(),
                'lr': lr,
                'step_size': step_size,
                'gamma': gamma,
                'weight_decay': weight_decay,
                'hidden_dim': hidden_dim,
                'num_layers': num_layers,
                'batch_size': batch_size,
                'frame_frequency': frame_frequency,
                'input_dim': input_dim,
                'num_classes': num_classes,
                'train_dataset': len(train_dataset),
                'test_dataset': len(test_dataset),
                'avg_loss_list': avg_loss_list,
                'avg_accuracy_list': avg_accuracy_list,
                'avg_test_accuracy_list': avg_test_accuracy_list,
                'avg_top5_test_accuracy_list': avg_top5_test_accuracy_list,
                'avg_test_loss_list': avg_test_loss_list},
                result_name)



In [ ]:
import shutil
def copy_ipynb_file(current_time): 
    current_file = 'dino_lstm_heatmap.ipynb'
    copy_file = '/home/osero/Desktop/CMPE/dinov2/classsification/lstm/lstm_results/ipynbs/COPY_' + current_time + '_' + current_file
    shutil.copy(current_file, copy_file)

## Train

In [28]:
lr = 0.0001
step_size = 5
gamma = 0.5
weight_decay = 0

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=lr)
scheduler = lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma) ## CosineAnnealingLR Dene
print(f"lr {lr}, step_size: {step_size}, gamma: {gamma}, weight_decay: {weight_decay}")
print(f"Model hidden_dim {hidden_dim}, num_layers: {num_layers}")
print(f"batch_size {batch_size}, frame_frequency: {frame_frequency}")

avg_loss_list = []
avg_accuracy_list = []
avg_test_accuracy_list = []
avg_top5_test_accuracy_list = []
avg_test_loss_list = []

num_epoch = 30
for epoch in range(num_epoch):
    train_acc = 0
    train_loss = 0
    loop = tqdm(train_loader)

    running_loss = 0.0
    running_accuracy= 0.0
    for idx, (features, labels) in enumerate(loop):
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)

        predictions = outputs.argmax(dim=1, keepdim=True).squeeze()
        correct = (predictions == labels).sum().item()
        accuracy = correct / batch_size

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_accuracy += 100 * accuracy
        loop.set_description(f"Epoch [{epoch}/{num_epoch}]")
        loop.set_postfix(loss=loss.item(), acc=accuracy)
    scheduler.step()
    avg_loss = running_loss / len(train_loader)
    avg_accuracy = running_accuracy / len(train_loader)
    print(f"Time: {get_current_time()} Epoch [{epoch}], Avg loss: {avg_loss:.4f}, Avg accuracy: {avg_accuracy:.4f}")
    avg_test_accuracy, avg_top5_test_accuracy, avg_test_loss = test_images()

    avg_loss_list.append(avg_loss)
    avg_accuracy_list.append(avg_accuracy)
    avg_test_accuracy_list.append(avg_test_accuracy)
    avg_top5_test_accuracy_list.append(avg_top5_test_accuracy)
    avg_test_loss_list.append(avg_test_loss)
current_time = get_current_time()
save_model_result(current_time)
copy_ipynb_file(current_time)

Epoch [0/30]: 100%|██████████| 1301/1301 [01:00<00:00, 21.47it/s, acc=1, loss=1.93]


Time: 2024-11-27_18-56-53 Epoch [0], Avg loss: 3.7569, Avg accuracy: 7.7633
Accuracy of the network on the 317 test video: 9.4637 %, top5: 29.3375 %, avg_loss: 3.4890566575414375


Epoch [1/30]: 100%|██████████| 1301/1301 [00:56<00:00, 23.08it/s, acc=0, loss=3.03] 


Time: 2024-11-27_18-57-58 Epoch [1], Avg loss: 3.2004, Avg accuracy: 14.2198
Accuracy of the network on the 317 test video: 18.6120 %, top5: 48.5804 %, avg_loss: 3.0062006762727203


Epoch [2/30]: 100%|██████████| 1301/1301 [00:58<00:00, 22.41it/s, acc=0, loss=3.65] 


...

Time: 2024-11-27_19-01-16 Epoch [4], Avg loss: 2.2870, Avg accuracy: 32.8209
Accuracy of the network on the 317 test video: 31.8612 %, top5: 73.1861 %, avg_loss: 2.254854745387279


Epoch [5/30]: 100%|██████████| 1301/1301 [00:56<00:00, 22.83it/s, acc=0, loss=2.65]  


Time: 2024-11-27_19-02-21 Epoch [5], Avg loss: 1.9660, Avg accuracy: 41.2759
Accuracy of the network on the 317 test video: 37.2240 %, top5: 78.8644 %, avg_loss: 2.093813596181689


Epoch [6/30]:  10%|▉         | 124/1301 [00:05<00:56, 21.01it/s, acc=1, loss=1.08] 


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
import torch

# summarize history for accuracy
plt.plot(avg_accuracy_list) 
plt.plot(avg_test_accuracy_list)
plt.plot(avg_top5_test_accuracy_list)
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['Train', 'Test', 'Test Top-5'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(avg_loss_list)
plt.plot(avg_test_loss_list)
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

## Test

In [ ]:

# test_images()

## Report

In [ ]:
# print(classification_report(test_labels, test_predicted, target_names=class_names))


In [ ]:
# cm = confusion_matrix(test_labels, test_predicted)
# df_cm = pd.DataFrame(
#     cm, 
#     index = class_names,
#     columns = class_names
# )
# df_cm

In [ ]:
# def show_confusion_matrix(confusion_matrix):
#     hmap = sns.heatmap(confusion_matrix, annot=True, fmt="d", cmap="Blues")
#     plt.ylabel("Surface Ground Truth")
#     plt.xlabel("Predicted Surface")
#     plt.legend()
    
# show_confusion_matrix(df_cm)